"return_estimation" is practically our Q function.

Its designed to take an initial state, use a policy to find various trajectories. While finding these trajectories, it keeps track of the return value for each state-action combination, more specifically the moving average of returns.


It takes "returns" as input to pass it to the policy function which finds the policy for state-action combinations.

In the earlier versions, it would use a single state as the initial state and find trajectories starting with it.
However that will lead us to calculate expected return for some states incorrectly. There will be many trajectories starting from an initial s0 where some states just doesnt show up. This will lead to lower returns for those states on average.

In the current version we use all the states as equally likely initial state, so we explore all states and calculate their returns.

This ensures that we get avg. return for each combination of state-action, no matter how unlikely a trajectory gets us there.

The function returns a dictionary of the form key: (state,action) -> exp[return]

Which is basically the Q function.

How it differs from the definition:

The ALGORITHM goes like this:
        -pick a state s
        -pick an action a
        -check where this (s,a) takes us, specifically which state, lets say its s'
        -continue finding consequent states using the optimal policy π(s')
        -Use Gt = Rt + γGt+1 to find the return, this is the Q(s,a)

So to do this calculation, we need a policy first. After that we simply pick a combination of s and a, find next state that leads and calculate return.

It is important here to note that, **IN PRACTICE**, we dont follow the algorithm to **CALCULATE Q**, we follow the algorithm to **ADJUST** it.

We start with Q(s,a)=0 for all state-action pair.
The Gt bit that was calculated tells us how much we need to **ADJUST** Q(s,a) ***BASED ON NEW Q(s,a)***


We can just dismiss all our worries about *calculating* Q(s,a) on the go, we dont actually calculate it analytically, thats the entire purpose of reinforcement learning.

In our current example, the terminal state is 1. All policy will be defined or designed as such that we dont move from the terminal state. so Q(1,a) = 100, which is the immediet reward and return for that state. Lets say our current policy is π.

Now imagine there are immediet surrounding states. For all the actions that takes those states to the terminating point, we can calculate Q(s,a) for all those states.

A state s1 can immedietly reach the terminating point for some action a1.This will create a return.
Q(s1,a1). There can be some other action that takes state s1 to some state other than the terminating state. From that it can follow the policy, walk around different states and then finally come back to the terminating state 1. 

So our agent experiences 2 scenarios:
                1. take the action a1 to immedietly reach terminating state
                2. take some other action a' which sends it to a different trajectory and eventually brings it to terminating state.

So if we think of our agent in state 2, accordint to our "world-map" or basically how states are connected to each other and allowed action:
                1.agent takes the action L, reaches terminating point, gets reward
                2. agent takes other action, 0 or R, follows policy, get reward and a trajectory follows.

In the beginning, we obviously have no idea what Q(s,a) is. So what we do is let agent explore the environment. We can initialize Q(s,a) in whatever way we want as we will slowly upgrade it.

We always assume our current policy is optimal. Since Bellman equation applies to optimal states, we use that to calculate Q(s,a) ANALYTICALLY for a given trajectory.
So when our agent is in state 2:
                                1. Action a immedietly takes it to the terminating state and we simply calculate Q(s,a) = Rs + γRf, Rf is reward at final/terminal state.This is our 
                                Q(s,a) , FOR THIS TRAJECTORY
                                2. Action a1 sends it to some other trajectory that terminates in some terminating state. We then calculate Q(s,a) as being Rs + γQ(s',a1).

But how are we calculating Q(s',a1)? When the action took us to the terminating state it was easy, just γ*Rf right? Well, remember how scenario 2 creates a trajectory? Well there has to be some state in the trajectory that directly leads to the terminal state, we can find Q value of that state right? Just like we did in scenario 1! So if we calculate the Q of the state right before terminating state, we can subsequently find the Q value of all the states before that!This backtracking of Q value IS what gives is the bellman formula!

So we can look at the Bellman formula in a new way!

Just think it like this:

When we take an action a in some state s, whatever state that results in is the TERMINAL STATE.
Suppose we are trying to calculate Q(s,a) for some state s and some action a.
Instead of thinking of trajectory or policy, we can just SHRINK our entire environment, FOLD all the states resulting in just 3 states, s1, s2, s3. These states are all the possible states that all the possible actions a1,a2,a3 can take us. THIS is our world now.

if there are actions a1, a2,a3 takes us to state s1,s2,s3 from state s, ALL those states are TERMINAL STATE with definite reward point, just like we have 100 for state 1! so the reward will be = Rs + γR1 for action a1 and such.

Q(s,a1) = Rs + γR1 
Q(s,a2) = Rs + γR2
Q(s,a3) = Rs + γR3 [This notation is a mess]

We are just calculating our returns for resulting in TERMINAL STATE!.

To reiterate this moment, if our environment is just one non-ternimating state s and 3 terminating state s1,s2,s3, we can ANALYTICALLY find all the Q(state,action) values. It will literally be Q(s1,a=any)= R1 and so on and Q(s,a1) = Rs + γR1 and so on.This is it! There is nothing to it!

Now we are ready to UNFOLD our environment one state at a time 😎. If s1 or s2 were actual terminating states,we can do everything analytically. Since they are not, well, we can take each state at a time and unfold it slowly. 

We can take state s1 whose current reward is R1. But we would have access to this R1, RETURN OF S1 (not reward) value if it ACTUALLY was the terminating state. so what do we do? We UNFOLD the state s1 and find the terminating states *cough* that surrounds s1, along with the immediet reward for s1 called r1. Suppose unfolding s1 gets us terminating states *cough* s'1 and s'2, related to a1 and a2, whose rewards are R1 and R2. so we can calculate Q(s1,a1) and so on as follows:

Q(s1,a1) = r1 + γR'1
Q(s1,a2) = r1 + γR'2 

r1 is our REWARD for s1 and those are our Q values for s1.

So, what is our R1 ? We faked having it when calculating Q(s,a1). Unfolding s1 gave us r1 sure but what is R1 ? What is the RETURN for this state? What is the REWARD if this was the terminating state?

**Calculating Q(s,a) from Q(s',a') or rather ADJUSTING IT:**

We have several options here:

**Monte Carlo** says Q(s,a) is just G(s,a). Even though there are multiple possible return for each actions, from s1, Monte Carlo just sees what action is ended up being taken and uses that. 
Since you are going through stuffs at the end of episodes, it really doesnt matter to look for Q(s1,a1) etc. 
If in an episode s1 comes after s and in s1 action a2 is being taken, adjust the Q(s,a) based on Q(s1,a2) since that was the sequence that occured, simple.

**SARSA** says, well, **wait** till state s1 takes some action a1 and check its stated Q(s1,a1) and adjust Q(s,a) based on that. We are not waiting for a new Q(s1,a1) being calculated. There are already some value (even at the very beginning its 0 and not None) and when s transitions to s1, we **wait** for s1 to transition using some action, a1 or a2 or a3 and then use the already stated Q values for that state action (s1,a1) pair.

**EXPECTED-SARSA** says, well, when s reaches state s1, what is the **EXPECTED** return **ACROSS ALL POSSIBLE ACTIONS**. Meaning we dont need to check which action is ended up being taken. We just check the next state, use a convulation of probability distribution π(a|s) and 
Q(s,a) to find the expected return for all. Then adjust the previous Q(s,a) according to this.


**Q learning** is where the Bellman Optimality Equation comes from.

We can write Q(s,a) as following,

for a defined policy π(s), Q(s,a) is,

                    Q(s,a) = Rs + γ * max_a' (Q(s',a')) [Bellman'S Optimality equation]

If this looks like a bit self referential, dont worry we will clear it out.

The equation is not meant to give us Q(s,a) analytically, but give us a model for it. When the agent acts **optimally**, it will maintain that relationship.**Hence it being the Bellman Optimality Equation**

**THIS IS NOT A FORMULA FOR Q(s,a)**
**THERE IS NO FORMULA FOR Q(s,a) in its absolute sense or form**

Rather its a **MODEL** for Q(s,a) when Q* is reached. Its basically the formula for Q*

So how is Q(s,a) adjusted based on Q learning?

Well, after leaving s by some action a, we have Q(s1,a1) and Q(s1,a2). To find the RETURN for coming to this state should be the higher among these values, right? state s came to s1 thinking this is a terminating state. We GO to the terminating states to GET maximum rewards, hence they are the terminating state. The terminating state for a rover is safe landing on the ground, for that it should get the maximum reward out of this state right? 

But so s1 needs to give state s its maximum reward for coming to s1.right now s1 can achieve two rewards, one for going to s'1 and another for going to s'2. It cant give what it doesnt have. so s1 must get the maximum reward to give it to s. so the standing reward for s1 will be:

                                                                Max[ Q(s1,a1), Q(s1,a2)]

So we can derive the formula for Q(s,a1) as:
                                                                Q(s,a) = Rs + γMax[ Q(s1,a1), Q(s1,a2)]


This should feel satisfying! We derived the goddamn Bellman equation!


**ALL OF THESE VALUES ARE STOCHASTICALLY CALCULATED NONE IS ANALYTICAL**


**WE SAMPLE ALL OF THESE VALUES**
**WE ADJUST OUR POLICY/ADJUST Q OR BOTH**
**WE SAMPLE AGAIN**


So our trajectory for calculating Q(s,a) is the following:

                        1.Find where (s,a) takes us
                        2.Use its standing reward if its the terminating state || Find its surrounding terminal states
                        3.Repeat until you actually find the terminating state. Then calculate from there (Gt)

So this creates a tree . . . . 

                                                s1   reward Rs
                                                |
                                                | a1
                                                |
                                                s -> termanting state, reward R', Q(s,a) = Rs + γR'
                                              / |  \
                                             /  |   \
                                            / a1|    \
                                        a2 /    |     \ a3
                                          /     |      \
                                         s1'    s2'     s3'
                                        / |  \  .       .
                                       /  |   \ .       .
                                      / a1|    \.       .
                                   a2/    |     \a3
                                    /     |      \
                                  T1     s1''     T2
                                          |
                                          |  a1
                                          |
                                         T3


To find Q(s,a), walk down the tree till we reach a final state T1 or T2 or T3, while maximizing our returns.

It will be a pain in the ass Just to construct a tree like this for realistic scenarios, let alone traverse them.

If we could just traverse the tree, we can analytically find Q(s,a).

Only if we could!

What we can actually do:

Lets see what we have first :

        1. We have environment mapping ie we always know what s' is given (s,a)
        2. We have a policy function, which can be a random probabily one or one with some level of guiding based on what we know (if state is edge of the building and action is "Forward", the policy will distribute a probability of 0, ideally).


Well, we can do a LOTS Of things to be honest! In our CURRENT CODE we implemented the Monte-Carlo! But lets see the monte-carlo method: 

        1. Apply action to the state
        2. Find resulting state
        3. Use policy, which is to **select the action with MOST Q value at an epsilon % percentage** to traverse the tree to the terminating point
        4. Calculate Q(s,a) (is basically G(s,a) ) for this trajectory, assuming the trajectory run IS the optimal run our policy is optimal and THIS is the actual Q(s,a)
        6. Repeat from 1 to 5 and then update Q(s,a) by small amount based on the incoming Q(s,a)

Notice that in step 3, we might not hit ALL the states. So the trajectory might move along a branch where max[Q(s,a)] is not found. Thats why we are keeping the moving average of the returns. Instead of calculating Q(s,a) directly , we are using E[Q(s,a)], by basically running trajectories all over the place.


The "spice" here is that, we not only use the monte-carlo method. We also update the Q(s,a) values based on returns using the following formula:

                                                        Q_new = Q_old +  α(Q_new-Q_old)


In conclusion, Q(s,a) is an estimation and the bellman formula is a model. We are not using Bellman Formula here.


In [1]:
import random
import numpy as np

from scipy.special import softmax






reward_1 = 100
reward_6 = 40

def step_left(state):
  if(state!=1):
    return state-1
  else:
    return -1

def step_right(state):
  if(state!=6):
    return state+1
  else:
    return -1

def step_null(state):
  return state

# print("\n\n")
# for i in range(1,7):
#   print(f" {step_left(i)} <- state {i} ")
#   print(f"state {i} -> {step_right(i)} ")
# print("\n\n")

def new_state(state,action):
  if(action == "L"):
    return step_left(state)
  elif(action == "R"):
    return step_right(state)
  else:
    return step_null(state)

def reward(s,s_dash):
  if(s_dash==1):
    return reward_1
  elif(s_dash==6):
    return reward_6
  else:
    return 0


#Finds a trajectory from starting state, using return to calculate policy. 
#When returns is None,policy is calculated to be even distribution for actions, 1/3 for each since there are 3 actions
#when given a returns dictionary, it passes that dictionary to decide policy
#it doesnt calculate any return or even reward
#it can (and does), however, use updated policy to find optimal paths
#doesnt have its own returns, it takes return into account when given
def sampler(state,returns=None):
  trajectory = []
  s_init = state
  count = 0
  

  while(s_init!=1 and s_init!=6):

    if returns is None:
      action = random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init),k=1)[0]
    
    else:
      action = random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init,returns),k=1)[0]

    #print(f"current state: {s_init} action:{action} new state: {new_state(s_init,action)} count: {count} policy dist: {Policy.policy_dist_static(s_init)}")
    #print(f"{[s_init,action]}")
    trajectory.append([s_init,action,0,0])
    s_init = new_state(s_init,action)
    count = count + 1
    if((s_init!=6 and s_init!=1) and count>20000):
      return None

  trajectory.append([s_init,random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init),k=1)[0],0,0])
  
  return trajectory


#given a trajectory, calculate Gt for each state
def calculate_return(trajectory, discount=0.5):
    # Calculate return using (current_state, next_state) for reward
    for i in range(len(trajectory) - 1, -1, -1):
        current_state = trajectory[i][0]
        if i < len(trajectory) - 1:
            next_state = trajectory[i + 1][0]
            trajectory[i][2] = reward(current_state, next_state)
            trajectory[i][3] = trajectory[i][2] + discount * trajectory[i + 1][3]
        else:
            # Terminal state: no next state, return is 0
            trajectory[i][2] = 0
            trajectory[i][3] = 0
    return trajectory
    # returned trajectory looks like [s, "action", reward, return]




**REWARD HAPPENS AT TRANSITION NOT AT STATE**

**TRANSITIONING TO SAME STATE FROM DIFFERENT STATE CAN GIVE DIFFERENT REWARD**

**RETURN IS THE CUMULATIVE DISCOUNTED SUMMATION FOR REWARD**


In [2]:


#This is practically our Q(s,a) function.
def return_estimation(start_state, num_episodes,rets=None,discount=0.5,show=False):
    returns = {}
    counts = {}

    for i in range(num_episodes):
      s0 = random.randint(1, 6)
      if rets is None:
        #("we are here")
        
        trajectory = sampler(s0)
        #(trajectory)
      else:
        trajectory = sampler(s0,rets)
      if show:
         print([t[0] for t in trajectory]) 
      if trajectory is not None:
        #print(trajectory)
        trajectory = calculate_return(trajectory,discount=discount)
        for state, action, r, Gt in trajectory:
            key = (state, action)
            if key in returns:
                counts[key] += 1
                # update running average
                returns[key] += (Gt - returns[key]) / counts[key]
            else:
                returns[key] = Gt
                counts[key] = 1
    log_returns_table(returns,"from return estimator",filename="function_return.txt")
    return returns




class Policy:
    zero_ret = {}
    def __init__(self,init_val=0,epsilon=0.9):
        self.returns = {}
        self.policy_probability ={}
        self.epsilon = epsilon
        a = ["L","0","R"]
        for i in range(1,7):
            for j in a:
                key = (i,j)
                self.returns[key] = init_val

        Policy.zero_ret = dict(self.returns) 
        for i in range(1,7):
            self.policy_probability[i] = self.policy_dist(i)


    def policy_dist(self,s,returns=None):
        dist =[]
        if returns == None:
          returns = self.returns
        
        Q1 = returns.get((s,"L"),0)
        Q2 = returns.get((s,"0"),0)
        Q3 = returns.get((s,"R"),0)

        Qv = np.array([Q1, Q2, Q3])
        one_hot = epsilon_one_hot(Qv,self.epsilon)
        #print("Hi from one hot")
        return one_hot


       
    def policy_dist_nonhot(self,s,returns=None,):
        
        dist =[]
        if returns == None:
          returns = self.returns
        
        Q1 = returns.get((s,"L"),0)
        Q2 = returns.get((s,"0"),0)
        Q3 = returns.get((s,"R"),0)
        Qv = [Q1,Q2,Q3]
        dist = softmax(Qv) 
        #print("Hi from non-hot")
        return dist
    @staticmethod
    def policy_dist_static(s,returns=None):
        
        dist =[]
        if returns == None:
          returns = Policy.zero_ret
        
        Q1 = returns.get((s,"L"),0)
        Q2 = returns.get((s,"0"),0)
        Q3 = returns.get((s,"R"),0)
        Qv = [Q1,Q2,Q3]
        dist = softmax(Qv) 
            
        return dist
    
    def update(self,returns):
       self.returns = returns
       for i in range(1,7):
            self.policy_probability[i] = self.policy_dist(i)
    
    def update2(self,returns):
      self.epsilon = max(0.01, self.epsilon * 0.99)

      returns_new = dict(self.returns)  # Start with a copy of old returns
      
      for (s,a), r_new in sorted(returns.items()):
        r_old = self.returns.get((s,a), 0)  # Default to 0 if not in old returns
        
        # Exponential moving average with alpha=0.9
        r_updated = r_old + (r_new - r_old) * 0.9
        returns_new[(s,a)] = r_updated

      self.returns = returns_new
      
      for i in range(1,7):
          self.policy_probability[i] = self.policy_dist(i)


def log_returns_table(returns, iteration, filename="return_evolution.txt"):
    """
    Logs the return values in a formatted table style for each iteration.

    Args:
        returns (dict): {(state, action): return_value}
        iteration (int): Current iteration or episode batch number
        filename (str): File name to append the log to
    """
    actions = ["L", "0", "R"]
    states = sorted(set(s for (s, _) in returns.keys()))

    with open(filename, "a") as f:
        f.write(f"Iteration number: {iteration}\n")
        f.write(f"{'State':<8}" + "".join([f"{a:^12}" for a in actions]) + "\n")

        for s in states:
            f.write(f"{s:<8}")
            for a in actions:
                val = returns.get((s, a), 0.0)
                f.write(f"{val:<12.4f}")
            f.write("\n")

        f.write("\n")  # Blank line between iterations




def epsilon_one_hot(Qv, epsilon=0.1):
    Qv = np.array(Qv)
    n = len(Qv)
    one_hot = np.zeros(n)
    
    if np.random.rand() < epsilon:
        idx = np.random.randint(n)
    else:
        idx = np.argmax(Qv)
    
    one_hot[idx] = 1
    return one_hot





In [3]:
# returns = return_estimation(4,100,discount=0.5)

# for(s,a),er in sorted(returns.items()):

#     print(f" Q: {s}  {a}  -> {er:.4f}\n")

In [4]:
trajectory = sampler(4)

returns = calculate_return(trajectory=trajectory,discount= 0.5)
    
for i in returns:
    print(i)

[4, 'R', 0, 3.125]
[5, 'L', 0, 6.25]
[4, 'L', 0, 12.5]
[3, '0', 0, 25.0]
[3, 'L', 0, 50.0]
[2, 'L', 100, 100.0]
[1, 'L', 0, 0]


In [5]:






def naive_RL(iterations=100,discount=0.5,init_state=3,update_2 = True):
  
  policyy = Policy(epsilon=1)
  print(f"Starting naive RL with discount value:{discount} initial_state:{init_state}")

  print("state:--- policy distribution:  L  0  R BEFORE ITERATION")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policyy.policy_dist(i)[0]}  {policyy.policy_dist(i)[1]}  {policyy.policy_dist(i)[2]}")


  returns = policyy.returns
  for i in range(iterations):
    
    
    returns = return_estimation(init_state, 1,returns,discount=discount)
    if not update_2:
      policyy.update(returns)
    else:
      log_returns_table(policyy.returns,policyy.epsilon)
      policyy.update2(returns)

  
  print(f"state:--- policy distribution:  L  0  R after all Iterations ")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policyy.policy_dist(i)[0]:0.4f}  {policyy.policy_dist(i)[1]:0.4f}  {policyy.policy_dist(i)[2]:0.4f}")

  print("\n\n\n")
  
  return policyy





In [6]:


#Using learning

#THIS IS OUR OPTIMAL POLICY, or rather OUR DISTRIBUTION

policy = Policy()
policy = naive_RL(4000,discount=0.5,init_state=4)


print("After dust is settled")

print("Returns for discount 0.5")
for i in range(1,7):
    print(f" {i}        policy distribution:  {policy.policy_dist(i)[0]:0.4f}  {policy.policy_dist(i)[1]:0.4f}  {policy.policy_dist(i)[2]:0.4f}")


  
  

Starting naive RL with discount value:0.5 initial_state:4
state:--- policy distribution:  L  0  R BEFORE ITERATION
 1        policy distribution:  0.0  0.0  1.0
 2        policy distribution:  0.0  0.0  0.0
 3        policy distribution:  0.0  0.0  1.0
 4        policy distribution:  0.0  0.0  0.0
 5        policy distribution:  0.0  0.0  0.0
 6        policy distribution:  0.0  0.0  0.0
state:--- policy distribution:  L  0  R after all Iterations 
 1        policy distribution:  1.0000  0.0000  0.0000
 2        policy distribution:  1.0000  0.0000  0.0000
 3        policy distribution:  1.0000  0.0000  0.0000
 4        policy distribution:  1.0000  0.0000  0.0000
 5        policy distribution:  0.0000  0.0000  1.0000
 6        policy distribution:  1.0000  0.0000  0.0000




After dust is settled
Returns for discount 0.5
 1        policy distribution:  1.0000  0.0000  0.0000
 2        policy distribution:  1.0000  0.0000  0.0000
 3        policy distribution:  1.0000  0.0000  0.0000
 

In [7]:
# #NOT Using learning



# policy = Policy()
# policy = naive_RL(2000,discount=0.5,init_state=4,update_2=False)


# print("After dust is settled")

# print("Returns for discount 0.5")
# for i in range(1,7):
#     print(f" {i}        policy distribution:  {policy.policy_dist(i)[0]:0.4f}  {policy.policy_dist(i)[1]:0.4f}  {policy.policy_dist(i)[2]:0.4f}")


  
  

In [8]:
track = []
for i in range(1000):
  tt = sampler(4,policy.returns)
  tt = [row[0] for row in tt]
  track.append(tt)


from collections import Counter
# Convert inner lists to tuples
track_tuples = [tuple(x) for x in track]
# Count occurrences
counts = Counter(track_tuples)
# Total number of elements
total = len(track)
# Calculate percentages
percentages = {key: (value / total) * 100 for key, value in counts.items()}
# Sort percentages by value descending and take top 3
top3 = sorted(percentages.items(), key=lambda x: x[1], reverse=True)[:]
# Print nicely
for element, pct in top3:
    print(f"{list(element)}: {pct:.2f}%")  # convert back to list if needed


[4, 3, 2, 1]: 99.00%
[4, 5, 6]: 1.00%


In [9]:
import numpy as np

values = np.array([12.5,10])
probs = np.exp(values) / np.sum(np.exp(values))
print(probs)




[0.92414182 0.07585818]
